In [19]:
import pandas as pd
df=pd.read_csv(r"C:\Users\Deepalakshmi\Downloads\sample_emails_with_triage_200.csv")

In [20]:
df.head()

,id,sender,subject,body,priority,triage_label,ideal_intent,ideal_tone
0,1,alerts@bank.com,Password Reset Request,Reminder: The client meeting is scheduled at 1...,low,notify_human,notify,urgent
1,2,alerts@bank.com,Congratulations! You've Won,Your invoice of INR 25515.09 is due on 2025-12...,low,respond,respond,polite
2,3,no-reply@service.com,Promotion: Big Sale,Reminder: The client meeting is scheduled at 1...,low,ignore,notify,urgent
3,4,sales@shop.com,Monthly Report,"Hello team, please find the attached weekly re...",medium,respond,respond,polite
4,5,no-reply@service.com,Survey,"Hello team, please find the attached weekly re...",low,respond,respond,polite


In [21]:
DANGEROUS_ACTIONS = ["respond"]

In [22]:
def hitl_check(action):
    if action in DANGEROUS_ACTIONS:
        return "WAIT_FOR_HUMAN" 
    return "AUTO_APPROVED"

In [26]:

#Simulate Human Approval
def human_decisions():
    decision = input("Approve action? (yes/no): ")
    return decision.lower() == "yes"    

In [24]:
def email_assistant(email_text):
    text = str(email_text).lower()

    # URGENT / NOTIFY
    if any(word in text for word in [
        "urgent", "asap", "immediately", "submit", "deadline", "eod", "today"
    ]):
        return "notify", "urgent"

    # RESPOND / POLITE
    elif any(word in text for word in [
        "please", "can you", "could you", "kindly", "review", "confirm", "reply"
    ]):
        return "respond", "polite"

    # IGNORE / POLITE
    elif any(word in text for word in [
        "thank you", "thanks", "appreciate", "grateful"
    ]):
        return "ignore", "polite"

    # INFORMATIONAL / NEUTRAL
    elif any(word in text for word in [
        "meeting", "schedule", "reminder", "update", "announcement"
    ]):
        return "notify", "neutral"

    # DEFAULT (UNCLEAR)
    else:
        return "review", "neutral"


In [25]:
results =[]
for _, row in df.sample(5).iterrows():
    action, tone= email_assistant(row["body"])
    status = hitl_check(action)

    if status == "WAIT_FOR_HUMAN":
        approved = human_decisions()
        final_action - action if approved else "blocked"
    else:
        final_action = action
    results.append({
        "email": row['body'][:50],
        "ai_action": action,
        "final_action": final_action,
        "hitl_status": status
    })
pd.DataFrame(results)

,email,ai_action,final_action,hitl_status
0,"Dear user, we detected a login from a new devi...",notify,notify,AUTO_APPROVED
1,"Hi, don't miss our sale with discounts up to 7...",review,review,AUTO_APPROVED
2,Your order #3621 has been shipped and is expec...,review,review,AUTO_APPROVED
3,Security alert: multiple failed login attempts...,review,review,AUTO_APPROVED
4,Reminder: The client meeting is scheduled at 1...,notify,notify,AUTO_APPROVED
